# RF-DETR-M без COCO-чекпойнта на 5 семейных фолдах (bot-5)
**Только честный замер. Ничего не идёт в сдачу до решения по весам / письма организаторов.**
Код: `scripts/rfdetr_train.py` (свой цикл; из rfdetr 1.11.2, Apache-2.0, — только сборка модели/матчер/критерий). Энкодер: `facebook/dinov2-small` (Apache-2.0), переносится вручную. **Никаких** `rf-detr-*.pth` (COCO/O365) не скачивается. Подробности — `kaggle/README_KAGGLE_RFDETR.md`.

Inputs: `expds-koz3-bot5` (репо, ветка `bot5-rfdetr-kaggle`), `koz3-spor-det05` (`f0..f4`, `grid_f0..grid_f4`). GPU T4×2, Internet ON (pip + HF dinov2-small).

In [ ]:
!pip install -q "rfdetr==1.11.2" --no-deps 2>&1 | tail -1
!pip install -q supervision pyDeprecate "pydantic>=2" 2>&1 | tail -1
import torch, rfdetr; print(torch.__version__, rfdetr.__version__ if hasattr(rfdetr,'__version__') else '', torch.cuda.device_count())

In [ ]:
import os
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь датасета
DET='/kaggle/input/koz3-spor-det05'
OUT='/kaggle/working/rfdetr'
os.makedirs(OUT, exist_ok=True)
from huggingface_hub import snapshot_download
DINO=snapshot_download('facebook/dinov2-small', allow_patterns=['*.json','*.safetensors','README.md'])
print(DINO); print(open(DINO+'/README.md').read()[:60])  # должно быть license: apache-2.0
!ls $DET

## Шаг 1 — smoke (10 итераций, обязателен первым). Только честный замер скорости/памяти.
Проверить в логе `dinov2: {'copied': 223, ... 'missing': 0}`; записать `s_per_iter` и `cuda_peak_MB` из timing.json. Smoke идёт с bs ≤ 2 и без накопления — `cuda_peak_MB` для bs 4 нужно пересчитать (или прогнать `--iters 20 --bs 4 --accum 1` вне smoke).

In [ ]:
!python $REPO/scripts/rfdetr_train.py --data $DET/f0 --out $OUT/smoke --model rfdetr-m --size 640 --fp16 --dinov2-dir $DINO --smoke
!cat $OUT/smoke/timing.json
!python $REPO/scripts/rfdetr_train.py --data $DET/f0 --out $OUT/mem_bs4 --model rfdetr-m --size 640 --fp16 --dinov2-dir $DINO --iters 20 --bs 4 --accum 1 --warmup-iters 5 --log-every 5 --pred-split none 2>&1 | tail -3

Бюджет полосы GPU0 (3 фолда): `s_per_iter(bs4) × accum 4 × ITERS × 3` + сетка. Если > ~11 ч — `ITERS=6000` или фолды по сессиям. OOM на bs 4 → `--bs 2 --accum 8`.

In [ ]:
ITERS=8000
!bash $REPO/kaggle/rfdetr_folds.sh $DET $OUT rfdetr-m $ITERS $DINO

In [ ]:
!for k in 0 1 2 3 4; do cat $OUT/f$k/timing.json | python -c "import json,sys;d=json.load(sys.stdin);print($k,{x:d.get(x) for x in ['s_per_iter','train_s','cuda_peak_MB','loss_first','loss_last']})"; done
!cd /kaggle/working && tar czf rfdetr_preds.tgz rfdetr/f*/grid/preds_valid.json rfdetr/f*/timing.json rfdetr/f*/grid/timing.json rfdetr/*.log && ls -la rfdetr_preds.tgz

Скачать `rfdetr_preds.tgz`. Дальше — на машине с данными: `README_KAGGLE_RFDETR.md`, шаг 4 (coco_preds_to_stack `--group sp_rfdetr` → scoreboard --honest 10 → П1–П6).